# Anchor — PaliGemma2 Multi-LoRA Inference

[![GitHub](https://img.shields.io/badge/GitHub-recursia--lab%2Fanchor-blue)](https://github.com/recursia-lab/anchor)

This notebook shows how to use [Anchor](https://github.com/recursia-lab/anchor) to run PaliGemma2 with multiple LoRA adapters — switching between them in ~216ms, all in VRAM.

**What you'll do:**
1. Install the `anchor-vision` client
2. Connect to a running Anchor endpoint
3. Run PCB defect detection with different adapters
4. (Optional) Use the LangChain integration

## 1. Install

In [ ]:
# Core client (stdlib only, no deps)
!pip install anchor-vision -q

# Optional: LangChain integration
!pip install 'anchor-vision[langchain]' -q

## 2. Connect to Anchor

In [ ]:
from anchor_vision import AnchorClient

# Replace with your Anchor endpoint
ENDPOINT = "https://anchor-jwp4m2y72q-uk.a.run.app"  # public demo

client = AnchorClient(ENDPOINT)

# Check what adapters are loaded
health = client.health()
print("Status:", health["status"])
print("Adapters:", health["adapters"])

## 3. Download a test image

In [ ]:
import urllib.request
from IPython.display import Image, display

# Download a sample PCB image (CC0 licensed)
url = "https://raw.githubusercontent.com/recursia-lab/anchor/main/examples/sample_pcb.jpg"
urllib.request.urlretrieve(url, "sample_pcb.jpg")
display(Image("sample_pcb.jpg", width=400))

## 4. Run inference with different adapters

In [ ]:
adapters = ["open_circuit", "short", "mouse_bite", "missing_hole", "spur"]
prompts = {
    "open_circuit": "Is there an open circuit defect? Answer YES or NO.",
    "short": "Is there a short circuit defect? Answer YES or NO.",
    "mouse_bite": "Is there a mouse bite defect? Answer YES or NO.",
    "missing_hole": "Is there a missing hole defect? Answer YES or NO.",
    "spur": "Is there a spur defect? Answer YES or NO.",
}

print(f"{'Adapter':<20} {'Answer':<8} {'Latency':>8}")
print("-" * 40)

for adapter in adapters:
    result = client.inspect(
        "sample_pcb.jpg",
        prompt=prompts[adapter],
        adapter=adapter,
    )
    latency = f"{result.latency_ms}ms" if result.latency_ms else "—"
    print(f"{adapter:<20} {result.answer:<8} {latency:>8}")

## 5. Use your own image

In [ ]:
from google.colab import files
uploaded = files.upload()

image_path = list(uploaded.keys())[0]

result = client.inspect(
    image_path,
    prompt="Describe any defects visible in this PCB image.",
    adapter="open_circuit",
    max_tokens=50,
)
print("Answer:", result.answer)
print(f"Latency: {result.latency_ms}ms")

## 6. LangChain integration (optional)

In [ ]:
from anchor_vision import AnchorVisionTool

tool = AnchorVisionTool(
    endpoint=ENDPOINT,
    adapter="open_circuit",
    prompt="Is there an open circuit defect? Answer YES or NO.",
)

# Use directly
result = tool.invoke({"image_path": "sample_pcb.jpg"})
print("LangChain result:", result)

# Or drop into any LangChain agent
# agent = initialize_agent(tools=[tool], ...)

## Next steps

- **Train your own adapter**: See [`examples/finetune.py`](https://github.com/recursia-lab/anchor/blob/main/examples/finetune.py)
- **Deploy Anchor**: See the [Quick Start](https://github.com/recursia-lab/anchor#quick-start) guide
- **Adapter Hub**: Browse community adapters at [recursia-lab/paligemma2-adapters](https://github.com/recursia-lab/paligemma2-adapters)
- **Star the repo**: [recursia-lab/anchor](https://github.com/recursia-lab/anchor) ⭐